导入依赖

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

依赖配置

In [2]:
torch.manual_seed(42069)
np.random.seed(42069)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42069)

加载数据并查看尺寸

In [3]:
train = np.load("timit_11/train_11.npy")
train_label = np.load("timit_11/train_label_11.npy")
test = np.load("timit_11/test_11.npy")

print(train.shape, train_label.shape, test.shape)


(1229932, 429) (1229932,) (451552, 429)


定义数据集类

In [7]:
class TIMITDataset(Dataset):
    def __init__(self, data, labels=None):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels.astype(int), dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.data[idx], self.labels[idx]
        else:
            return self.data[idx]

划分训练集和验证集，比例为0.8:0.2

In [8]:
num_of_train = int(len(train) * 0.8)
train_dataset = TIMITDataset(train[:num_of_train], train_label[:num_of_train])
validation_dataset = TIMITDataset(train[num_of_train:], train_label[num_of_train:])
test_dataset = TIMITDataset(test[:])
print(len(train_dataset), len(validation_dataset), len(test_dataset))


983945 245987 451552


数据分批加载

In [9]:
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

获取输出标签的数量

In [10]:
# 获取所有不同的标签
unique_labels = np.unique(train_label)
# 计算不同标签的数量
num_classes = len(unique_labels)
print(f"共有 {num_classes} 个不同的标签。")

共有 39 个不同的标签。


定义网络结构

In [11]:
class Classifier(nn.Module):
    def __init__(self):
        """
        429个输入特征，39个输出标签
        """
        super().__init__()
        self.net = nn.Sequential(#线性->批归一化->ReLU激活函数->nn.Dropout(0.25)
            nn.Linear(429,2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 39),
        )
    def forward(self, x):
        out = self.net(x)
        return out

训练模型并在每一轮训练后验证并保存最佳模型

In [12]:
EPOCHES = 40
model = Classifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

best_acc = 0.0

for epoch in range(EPOCHES):
    model.train()
    total_loss = 0
    for data, labels in train_loader:
        data, labels = data.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHES}], Training Loss: {avg_loss}", end=", ")

    # 验证模型
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, labels in validation_loader:
            data, labels = data.to(device), labels.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    print(f"Validation Accuracy: {val_acc}")

    # 若精度提升，则保存模型
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"--> Saved best model with Accuracy: {best_acc}")


Epoch [1/40], Training Loss: 1.3279, Validation Accuracy: 0.6909
--> Saved best model with Accuracy: 0.6909
Epoch [2/40], Training Loss: 1.0653, Validation Accuracy: 0.7115
--> Saved best model with Accuracy: 0.7115
Epoch [3/40], Training Loss: 0.9872, Validation Accuracy: 0.7225
--> Saved best model with Accuracy: 0.7225
Epoch [4/40], Training Loss: 0.9370, Validation Accuracy: 0.7275
--> Saved best model with Accuracy: 0.7275
Epoch [5/40], Training Loss: 0.8990, Validation Accuracy: 0.7328
--> Saved best model with Accuracy: 0.7328
Epoch [6/40], Training Loss: 0.8670, Validation Accuracy: 0.7369
--> Saved best model with Accuracy: 0.7369
Epoch [7/40], Training Loss: 0.8409, Validation Accuracy: 0.7382
--> Saved best model with Accuracy: 0.7382
Epoch [8/40], Training Loss: 0.8178, Validation Accuracy: 0.7417
--> Saved best model with Accuracy: 0.7417
Epoch [9/40], Training Loss: 0.7971, Validation Accuracy: 0.7417
Epoch [10/40], Training Loss: 0.7780, Validation Accuracy: 0.7456
--> S

测试集预测并保存结果到CSV文件

In [13]:
# 加载最佳模型进行测试
model.load_state_dict(torch.load('best_model.pth'))
correct = 0
total = 0
predict = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        outputs = model(data)
        _, test_predict = torch.max(outputs.data, 1)
        for y in test_predict.cpu().numpy():
            predict.append(y)

with open('prediction.csv', 'w') as f:
    f.write('Id,Class\n')
    for i, y in enumerate(predict):
        f.write(f"{i},{y}\n")